# MapWorkIds — location_work_ids maintenance (oxjob #764)

Nightly identity step, runs between `Locations_with_Types` and `Locations_Mapped`.
Resolves or mints a work_id for every anchor `(provenance, namespace, native_id)`
not yet pinned in the registry. Insert-only for existing anchors: a non-NULL
registry work_id is never changed here — repoints are explicit sweep operations.
`work_id_map` keeps clustering + minting; consumers read only its frozen `work_id`
column (minted from `id`, repointed on displacement; legacy paper_id adoption retired and the column dropped — oxjob #764).

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.work_id_map') (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 6600000001 INCREMENT BY 1),
  doi STRING,
  pmid STRING,
  arxiv STRING,
  title_author STRING,
  work_id_source STRING,
  published_date DATE,
  openalex_created_dt DATE,
  openalex_updated_dt TIMESTAMP,
  work_id BIGINT
)
CLUSTER BY (doi, pmid, arxiv, title_author)
TBLPROPERTIES (
  'delta.deletedFileRetentionDuration' = '60 days',
  'delta.logRetentionDuration' = '60 days',
  'delta.checkpoint.writeStatsAsJson' = 'false',
  'delta.checkpoint.writeStatsAsStruct' = 'true',
  'delta.enableDeletionVectors' = 'true',
  'delta.feature.deletionVectors' = 'supported',
  'delta.feature.rowTracking' = 'supported',
  'delta.feature.v2Checkpoint' = 'supported')

In [0]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.location_work_ids') (
  provenance STRING,
  native_id_namespace STRING,
  native_id STRING,
  work_id BIGINT,
  work_id_source STRING,
  openalex_created_dt DATE,
  openalex_updated_dt TIMESTAMP,
  seeded_dt DATE,
  seeded_from STRING
)
CLUSTER BY (provenance, native_id_namespace, native_id)

In [ ]:
CREATE TABLE IF NOT EXISTS identifier('openalex' || :env_suffix || '.works.location_work_ids_run_stats') (
  run_ts TIMESTAMP,
  run_dt DATE,
  anchors_new_resolved BIGINT,
  anchors_new_null BIGINT,
  anchors_retried_resolved BIGINT,
  anchors_retried_null BIGINT,
  src_doi BIGINT,
  src_pmid BIGINT,
  src_arxiv BIGINT,
  src_title_author BIGINT,
  src_mag_legacy BIGINT,
  src_pmh_legacy BIGINT,
  minted_range BIGINT,
  distinct_works BIGINT,
  max_work_fanin BIGINT,
  top_fanin_work_id BIGINT,
  registry_null_backlog BIGINT,
  mints_displaced BIGINT,
  bridge_conflicts BIGINT
)

## Pending anchors → mint candidates
Pending = w_types anchors with no registry entry, plus registry entries still NULL
(NULL anchors retry nightly until they pin).

In [ ]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
CLUSTER BY (doi, pmid, arxiv, title_author)
AS
WITH t AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY provenance, native_id_namespace, native_id
           ORDER BY updated_date DESC) AS rwcnt
  FROM identifier('openalex' || :env_suffix || '.works.locations_w_types')
  QUALIFY rwcnt = 1
),
pending AS (
  SELECT t.*
  FROM t
  LEFT JOIN identifier('openalex' || :env_suffix || '.works.location_work_ids') r
    ON  t.provenance = r.provenance
    AND t.native_id_namespace = r.native_id_namespace
    AND t.native_id = r.native_id
  WHERE r.native_id IS NULL OR r.work_id IS NULL
),
-- degenerate keys ('' / bare 'arXiv:') must never become match keys
keys AS (
  SELECT NULLIF(merge_key.doi, '')          AS doi,
         NULLIF(merge_key.pmid, '')         AS pmid,
         CASE WHEN merge_key.arxiv IN ('arXiv:', '') THEN NULL
              ELSE merge_key.arxiv END      AS arxiv,
         NULLIF(merge_key.title_author, '') AS title_author,
         published_date
  FROM pending
)
SELECT
  doi, pmid, arxiv, title_author,
  (
    IF(doi IS NOT NULL, 4, 0) +
    IF(pmid IS NOT NULL, 3, 0) +
    IF(arxiv IS NOT NULL, 2, 0) +
    IF(title_author IS NOT NULL, 1, 0)
  ) AS key_score,
  MIN(published_date) AS published_date,
  CAST(:run_date AS DATE) AS openalex_created_dt,
  current_timestamp() AS openalex_updated_dt
FROM keys
GROUP BY doi, pmid, arxiv, title_author

### Mint into `work_id_map` — `DOI` key

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT regexp_replace(doi, '[^a-zA-Z0-9\./-]', '') as cleaned_doi, *
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
  WHERE doi IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY regexp_replace(doi, '[^a-zA-Z0-9\./-]', '')
    ORDER BY key_score DESC, openalex_updated_dt DESC
  ) = 1
) AS source
ON regexp_replace(target.doi, '[^a-zA-Z0-9\./-]', '') = cleaned_doi
WHEN MATCHED
AND (
  target.doi IS DISTINCT FROM COALESCE(source.doi, target.doi) OR
  target.pmid IS DISTINCT FROM COALESCE(source.pmid, target.pmid) OR
  target.arxiv IS DISTINCT FROM COALESCE(source.arxiv, target.arxiv) OR
  target.title_author IS DISTINCT FROM COALESCE(source.title_author, target.title_author)
)
THEN UPDATE SET
  target.doi = COALESCE(source.doi, target.doi),
  target.pmid = COALESCE(source.pmid, target.pmid),
  target.arxiv = COALESCE(source.arxiv, target.arxiv),
  target.title_author = COALESCE(source.title_author, target.title_author),
  target.openalex_created_dt = LEAST(target.openalex_created_dt, source.openalex_created_dt),
  target.openalex_updated_dt = source.openalex_updated_dt,
  target.work_id_source = 'doi'
WHEN NOT MATCHED THEN INSERT (
  doi, pmid, arxiv, title_author,
  openalex_created_dt, openalex_updated_dt, work_id_source
) VALUES (
  source.doi, source.pmid, source.arxiv, source.title_author,
  source.openalex_created_dt, source.openalex_updated_dt, 'doi'
)

### Mint into `work_id_map` — `PMID` key

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT *
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
  WHERE doi IS NULL AND pmid IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY pmid
    ORDER BY key_score DESC, openalex_updated_dt DESC
  ) = 1
) AS source
ON target.pmid = source.pmid
WHEN MATCHED
AND (
  target.doi IS DISTINCT FROM COALESCE(source.doi, target.doi) OR
  target.pmid IS DISTINCT FROM COALESCE(source.pmid, target.pmid) OR
  target.arxiv IS DISTINCT FROM COALESCE(source.arxiv, target.arxiv) OR
  target.title_author IS DISTINCT FROM COALESCE(source.title_author, target.title_author)
)
THEN UPDATE SET
  target.doi = COALESCE(source.doi, target.doi),
  target.pmid = COALESCE(source.pmid, target.pmid),
  target.arxiv = COALESCE(source.arxiv, target.arxiv),
  target.title_author = COALESCE(source.title_author, target.title_author),
  target.openalex_created_dt = LEAST(target.openalex_created_dt, source.openalex_created_dt),
  target.openalex_updated_dt = source.openalex_updated_dt,
  target.work_id_source = 'pmid'
WHEN NOT MATCHED THEN INSERT (
  doi, pmid, arxiv, title_author,
  openalex_created_dt, openalex_updated_dt, work_id_source
) VALUES (
  source.doi, source.pmid, source.arxiv, source.title_author,
  source.openalex_created_dt, source.openalex_updated_dt, 'pmid'
)

### Mint into `work_id_map` — `ARXIV` key

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT *
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
  WHERE doi IS NULL AND pmid IS NULL AND arxiv IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY arxiv
    ORDER BY key_score DESC, openalex_updated_dt DESC
  ) = 1
) AS source
ON target.arxiv = source.arxiv
WHEN MATCHED
AND (
  target.doi IS DISTINCT FROM COALESCE(source.doi, target.doi) OR
  target.pmid IS DISTINCT FROM COALESCE(source.pmid, target.pmid) OR
  target.arxiv IS DISTINCT FROM COALESCE(source.arxiv, target.arxiv) OR
  target.title_author IS DISTINCT FROM COALESCE(source.title_author, target.title_author)
)
THEN UPDATE SET
  target.doi = COALESCE(source.doi, target.doi),
  target.pmid = COALESCE(source.pmid, target.pmid),
  target.arxiv = COALESCE(source.arxiv, target.arxiv),
  target.title_author = COALESCE(source.title_author, target.title_author),
  target.openalex_created_dt = LEAST(target.openalex_created_dt, source.openalex_created_dt),
  target.openalex_updated_dt = source.openalex_updated_dt,
  target.work_id_source = 'arxiv'
WHEN NOT MATCHED THEN INSERT (
  doi, pmid, arxiv, title_author,
  openalex_created_dt, openalex_updated_dt, work_id_source
) VALUES (
  source.doi, source.pmid, source.arxiv, source.title_author,
  source.openalex_created_dt, source.openalex_updated_dt, 'arxiv'
)

### Mint into `work_id_map` — `TITLE_AUTHOR` key

In [0]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT *
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map_new_candidates')
  WHERE doi IS NULL AND pmid IS NULL AND arxiv IS NULL AND title_author IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY title_author
    ORDER BY key_score DESC, openalex_updated_dt DESC
  ) = 1
) AS source
ON target.title_author = source.title_author
WHEN MATCHED
AND (
  target.doi IS DISTINCT FROM COALESCE(source.doi, target.doi) OR
  target.pmid IS DISTINCT FROM COALESCE(source.pmid, target.pmid) OR
  target.arxiv IS DISTINCT FROM COALESCE(source.arxiv, target.arxiv) OR
  target.title_author IS DISTINCT FROM COALESCE(source.title_author, target.title_author)
)
THEN UPDATE SET
  target.doi = COALESCE(source.doi, target.doi),
  target.pmid = COALESCE(source.pmid, target.pmid),
  target.arxiv = COALESCE(source.arxiv, target.arxiv),
  target.title_author = COALESCE(source.title_author, target.title_author),
  target.openalex_created_dt = LEAST(target.openalex_created_dt, source.openalex_created_dt),
  target.openalex_updated_dt = source.openalex_updated_dt,
  target.work_id_source = 'title_author'
WHEN NOT MATCHED THEN INSERT (
  doi, pmid, arxiv, title_author,
  openalex_created_dt, openalex_updated_dt, work_id_source
) VALUES (
  source.doi, source.pmid, source.arxiv, source.title_author,
  source.openalex_created_dt, source.openalex_updated_dt, 'title_author'
)

In [0]:
-- Frozen work_id for freshly minted rows
UPDATE identifier('openalex' || :env_suffix || '.works.work_id_map')
SET work_id = id
WHERE work_id IS NULL

## Resolve pending anchors and upsert the registry
Verdicts are materialized first (`location_work_ids_verdicts`, one run's pending anchors),
then MERGEd insert-only into the registry. Precedence: **established works first, tier
order doi → pmid → arxiv → title_author as the tiebreak** — a work that existed before
this run wins over a fresh mint on a higher tier. Fresh mints are used only when nothing
established matches. "Established" is anchored to `:run_date` (job parameter, defaults to
the trigger date), shared with the mint cells, so a run straddling midnight cannot count
its own mints as established. Guards: title_author length > 20, ≤ 3 distinct work_ids per
title_author key, degenerate keys excluded. Legacy adoption (mag id, pmh id) applies only
when the resolution would otherwise be a fresh mint. After the registry MERGE, any mint
displaced by an established or adopted verdict is repointed in `work_id_map` itself, so
the losing mint cannot become "established" tomorrow and capture same-key siblings — the
map's `work_id` is the frozen *verdict*, initially `COALESCE(paper_id, id)`, and a
displaced mint's verdict is the work that displaced it.

In [ ]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.location_work_ids_verdicts')
CLUSTER BY (provenance, native_id_namespace, native_id)
AS
WITH t AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY provenance, native_id_namespace, native_id
           ORDER BY updated_date DESC) AS rwcnt
  FROM identifier('openalex' || :env_suffix || '.works.locations_w_types')
  QUALIFY rwcnt = 1
),
pending AS (
  SELECT t.provenance, t.native_id_namespace, t.native_id, t.merge_key, t.ids,
         (r.native_id IS NOT NULL) AS is_retry
  FROM t
  LEFT JOIN identifier('openalex' || :env_suffix || '.works.location_work_ids') r
    ON  t.provenance = r.provenance
    AND t.native_id_namespace = r.native_id_namespace
    AND t.native_id = r.native_id
  WHERE r.native_id IS NULL OR r.work_id IS NULL
),
-- v_est = MIN over rows that existed before today (established); v_any also admits tonight's mints
d AS (
  SELECT doi AS k, MIN(work_id) AS v_any,
         MIN(CASE WHEN openalex_created_dt < CAST(:run_date AS DATE) THEN work_id END) AS v_est
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map')
  WHERE doi IS NOT NULL AND doi <> '' AND work_id IS NOT NULL GROUP BY doi
),
p AS (
  SELECT pmid AS k, MIN(work_id) AS v_any,
         MIN(CASE WHEN openalex_created_dt < CAST(:run_date AS DATE) THEN work_id END) AS v_est
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map')
  WHERE pmid IS NOT NULL AND pmid <> '' AND work_id IS NOT NULL GROUP BY pmid
),
a AS (
  SELECT arxiv AS k, MIN(work_id) AS v_any,
         MIN(CASE WHEN openalex_created_dt < CAST(:run_date AS DATE) THEN work_id END) AS v_est
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map')
  WHERE arxiv IS NOT NULL AND arxiv NOT IN ('arXiv:', '') AND work_id IS NOT NULL GROUP BY arxiv
),
ta AS (
  SELECT title_author AS k, MIN(work_id) AS v_any,
         MIN(CASE WHEN openalex_created_dt < CAST(:run_date AS DATE) THEN work_id END) AS v_est
  FROM identifier('openalex' || :env_suffix || '.works.work_id_map')
  WHERE title_author IS NOT NULL AND title_author <> '' AND work_id IS NOT NULL
  GROUP BY title_author
  HAVING COUNT(DISTINCT work_id) <= 3
),
pmh_mapping AS (
  SELECT LOWER(pmh_id) AS pmh_id, MIN(work_id) AS legacy_work_id
  FROM openalex.works_poc.work_id_to_pmh_id_final
  GROUP BY LOWER(pmh_id)
),
resolved AS (
  SELECT
    pd.provenance, pd.native_id_namespace, pd.native_id, pd.is_retry,
    COALESCE(d.v_est, p.v_est, a.v_est,
             CASE WHEN LENGTH(pd.merge_key.title_author) > 20 THEN ta.v_est END) AS established_work_id,
    COALESCE(d.v_any, p.v_any, a.v_any,
             CASE WHEN LENGTH(pd.merge_key.title_author) > 20 THEN ta.v_any END) AS fallback_work_id,
      SIZE(ARRAY_DISTINCT(FILTER(ARRAY(d.v_est, p.v_est, a.v_est), x -> x IS NOT NULL))) > 1
        AS established_conflict,
    CASE
      WHEN d.v_est IS NOT NULL THEN 'doi'
      WHEN p.v_est IS NOT NULL THEN 'pmid'
      WHEN a.v_est IS NOT NULL THEN 'arxiv'
      WHEN LENGTH(pd.merge_key.title_author) > 20 AND ta.v_est IS NOT NULL THEN 'title_author'
      WHEN d.v_any IS NOT NULL THEN 'doi'
      WHEN p.v_any IS NOT NULL THEN 'pmid'
      WHEN a.v_any IS NOT NULL THEN 'arxiv'
      WHEN LENGTH(pd.merge_key.title_author) > 20 AND ta.v_any IS NOT NULL THEN 'title_author'
    END AS base_source,
    CAST(get(filter(pd.ids, x -> x.namespace = 'mag').id, 0) AS BIGINT) AS mag_id
  FROM pending pd
  LEFT JOIN d  ON pd.merge_key.doi = d.k
  LEFT JOIN p  ON pd.merge_key.pmid = p.k
  LEFT JOIN a  ON pd.merge_key.arxiv = a.k
  LEFT JOIN ta ON pd.merge_key.title_author = ta.k
)
SELECT
  rs.provenance, rs.native_id_namespace, rs.native_id, rs.is_retry,
  rs.fallback_work_id, rs.established_conflict,
  CASE
    WHEN rs.established_work_id IS NULL AND rs.fallback_work_id IS NOT NULL
         AND rs.provenance = 'mag' AND rs.mag_id IS NOT NULL THEN rs.mag_id
    WHEN rs.established_work_id IS NULL AND rs.fallback_work_id IS NOT NULL
         AND rs.provenance IN ('repo', 'repo_backfill')
         AND pm.legacy_work_id IS NOT NULL THEN pm.legacy_work_id
    ELSE COALESCE(rs.established_work_id, rs.fallback_work_id)
  END AS work_id,
  CASE
    WHEN rs.established_work_id IS NULL AND rs.fallback_work_id IS NOT NULL
         AND rs.provenance = 'mag' AND rs.mag_id IS NOT NULL THEN 'mag_legacy'
    WHEN rs.established_work_id IS NULL AND rs.fallback_work_id IS NOT NULL
         AND rs.provenance IN ('repo', 'repo_backfill')
         AND pm.legacy_work_id IS NOT NULL THEN 'pmh_legacy'
    ELSE rs.base_source
  END AS work_id_source,
  current_timestamp() AS run_ts
FROM resolved rs
LEFT JOIN pmh_mapping pm
  ON rs.provenance IN ('repo', 'repo_backfill') AND LOWER(rs.native_id) = pm.pmh_id

In [ ]:
MERGE INTO identifier('openalex' || :env_suffix || '.works.location_work_ids') AS target
USING identifier('openalex' || :env_suffix || '.works.location_work_ids_verdicts') AS source
ON  target.provenance = source.provenance
AND target.native_id_namespace = source.native_id_namespace
AND target.native_id = source.native_id
WHEN MATCHED AND target.work_id IS NULL AND source.work_id IS NOT NULL
THEN UPDATE SET
  target.work_id = source.work_id,
  target.work_id_source = source.work_id_source,
  target.openalex_updated_dt = source.run_ts
WHEN NOT MATCHED THEN INSERT (
  provenance, native_id_namespace, native_id, work_id, work_id_source,
  openalex_created_dt, openalex_updated_dt, seeded_dt, seeded_from
) VALUES (
  source.provenance, source.native_id_namespace, source.native_id,
  source.work_id, source.work_id_source,
  CAST(:run_date AS DATE), source.run_ts, NULL, NULL
)

In [ ]:
-- A mint displaced by an established or adopted verdict must not survive as a live
-- verdict for its keys: repoint the map row so tomorrow's same-key anchors converge.
MERGE INTO identifier('openalex' || :env_suffix || '.works.work_id_map') AS target
USING (
  SELECT fallback_work_id AS displaced_id, MIN(work_id) AS chosen_id
  FROM identifier('openalex' || :env_suffix || '.works.location_work_ids_verdicts')
  WHERE work_id IS NOT NULL AND fallback_work_id IS NOT NULL
    AND work_id <> fallback_work_id
  GROUP BY fallback_work_id
) AS source
ON  target.work_id = source.displaced_id
AND target.openalex_created_dt >= CAST(:run_date AS DATE)
WHEN MATCHED THEN UPDATE SET
  target.work_id = source.chosen_id,
  target.openalex_updated_dt = current_timestamp()

## Per-run match-quality stats
Computed from the materialized verdicts table (pending-anchor scale), never inferred from
registry timestamps. New vs retried anchors are split so the per-run miss rate is honest.
`max_work_fanin` is the misbinding tripwire: many anchors collapsing onto one work in a
single run means a degenerate key slipped through. `mints_displaced` counts the nightly mint-lost-to-established corrections;
`bridge_conflicts` counts records whose strong keys (doi/pmid/arxiv) disagreed between two
already-established works — the silent pick-a-side class feeding the bridge-merge backlog.
Idempotent on `run_ts`; a run with no pending anchors logs nothing.

In [ ]:
INSERT INTO identifier('openalex' || :env_suffix || '.works.location_work_ids_run_stats')
  (run_ts, run_dt, anchors_new_resolved, anchors_new_null,
   anchors_retried_resolved, anchors_retried_null,
   src_doi, src_pmid, src_arxiv, src_title_author, src_mag_legacy, src_pmh_legacy,
   minted_range, distinct_works, max_work_fanin, top_fanin_work_id, registry_null_backlog,
   mints_displaced, bridge_conflicts)
WITH f AS (
  SELECT work_id, COUNT(*) AS n
  FROM identifier('openalex' || :env_suffix || '.works.location_work_ids_verdicts')
  WHERE work_id IS NOT NULL
  GROUP BY work_id ORDER BY n DESC LIMIT 1
)
SELECT
  MAX(v.run_ts),
  CAST(MAX(v.run_ts) AS DATE),
  SUM(CASE WHEN NOT v.is_retry AND v.work_id IS NOT NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN NOT v.is_retry AND v.work_id IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.is_retry AND v.work_id IS NOT NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.is_retry AND v.work_id IS NULL THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'doi' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'pmid' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'arxiv' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'title_author' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'mag_legacy' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id_source = 'pmh_legacy' THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.work_id > 6600000000 THEN 1 ELSE 0 END),
  COUNT(DISTINCT v.work_id),
  MAX(f.n),
  MAX(f.work_id),
  (SELECT COUNT(*) - COUNT(work_id)
   FROM identifier('openalex' || :env_suffix || '.works.location_work_ids')),
  SUM(CASE WHEN v.work_id IS NOT NULL AND v.fallback_work_id IS NOT NULL
            AND v.work_id <> v.fallback_work_id THEN 1 ELSE 0 END),
  SUM(CASE WHEN v.established_conflict THEN 1 ELSE 0 END)
FROM identifier('openalex' || :env_suffix || '.works.location_work_ids_verdicts') v
LEFT JOIN f ON TRUE
HAVING MAX(v.run_ts) IS NOT NULL
   AND NOT EXISTS (
     SELECT 1 FROM identifier('openalex' || :env_suffix || '.works.location_work_ids_run_stats') s
     WHERE s.run_ts = MAX(v.run_ts))

In [0]:
SELECT format_number(COUNT(*), 0) AS registry_anchors,
       format_number(COUNT(work_id), 0) AS with_work_id
FROM identifier('openalex' || :env_suffix || '.works.location_work_ids')